In [1]:
%pip install flask joblib

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [1]:
import os

print("Model exists:", os.path.exists("sentiment_model.pkl"))
print("Vectorizer exists:", os.path.exists("tfidf_vectorizer.pkl"))

Model exists: True
Vectorizer exists: True


In [2]:
import os

os.makedirs("templates", exist_ok=True)
os.makedirs("static", exist_ok=True)

print("Website folders created!")

Website folders created!


In [3]:
%%writefile app.py

from flask import Flask, request, jsonify, render_template
import joblib

app = Flask(__name__)

# Load your trained model and TF-IDF vectorizer
model = joblib.load("sentiment_model.pkl")
tfidf = joblib.load("tfidf_vectorizer.pkl")


@app.route("/")
def home():
    return render_template("index.html")


@app.route("/predict", methods=["POST"])
def predict():
    data = request.get_json(silent=True) or {}
    review = data.get("review", "")

    if not isinstance(review, str) or not review.strip():
        return jsonify({"error": "Please enter a movie review."}), 400

    # Convert the review into TF-IDF features
    review_tfidf = tfidf.transform([review])

    # Predict sentiment
    prediction = model.predict(review_tfidf)[0]

    # Calculate prediction probabilities
    probabilities = model.predict_proba(review_tfidf)[0]

    probability_scores = {
        str(label): round(float(prob) * 100, 2)
        for label, prob in zip(model.classes_, probabilities)
    }

    return jsonify({
        "sentiment": str(prediction),
        "probabilities": probability_scores
    })


if __name__ == "__main__":
    app.run(debug=False, port=5000)

Writing app.py


In [4]:
%%writefile templates/index.html

<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">

    <title>SentimentScope | Movie Review Analyzer</title>

    <link rel="stylesheet"
          href="{{ url_for('static', filename='style.css') }}">
</head>

<body>

    <header class="navbar">
        <div class="logo">Sentiment<span>Scope</span></div>
        <div class="tagline">AI-Powered Review Analysis</div>
    </header>

    <main class="container">

        <section class="hero">
            <p class="eyebrow">MACHINE LEARNING PROJECT</p>

            <h1>Understand the <span>feeling</span><br>behind every review.</h1>

            <p class="description">
                Enter a movie review and let our sentiment analysis model
                predict whether it expresses a positive or negative opinion.
            </p>
        </section>

        <section class="analyzer">

            <label for="review">Your movie review</label>

            <textarea
                id="review"
                maxlength="10000"
                placeholder="Example: The acting was brilliant and the story was captivating..."
            ></textarea>

            <div class="input-footer">
                <span id="charCount">0 / 10000 characters</span>
                <button id="analyzeBtn" onclick="analyzeReview()">
                    Analyze sentiment →
                </button>
            </div>

            <div class="examples">
                <span>Try an example:</span>

                <button onclick="setReview('The movie was fantastic, emotional and beautifully directed.')">
                    Positive review
                </button>

                <button onclick="setReview('The movie was boring, predictable and poorly acted.')">
                    Negative review
                </button>
            </div>

            <div id="result" class="result hidden" aria-live="polite">
                <p class="result-label">PREDICTED SENTIMENT</p>
                <h2 id="sentiment"></h2>
                <p id="probabilities"></p>
            </div>

        </section>

        <section class="info">
            <div class="info-card">
                <h3>89.88%</h3>
                <p>Test-set accuracy</p>
            </div>

            <div class="info-card">
                <h3>Logistic Regression</h3>
                <p>Best-performing tested model</p>
            </div>

            <div class="info-card">
                <h3>TF-IDF</h3>
                <p>Text feature extraction</p>
            </div>
        </section>

        <footer>
            <p>
                SentimentScope · A machine learning project
            </p>
            <p>
                Predictions are generated by a trained model and may be incorrect.
            </p>
        </footer>

    </main>

    <script src="{{ url_for('static', filename='script.js') }}"></script>

</body>
</html>

Writing templates/index.html


In [5]:
%%writefile static/style.css

* {
    box-sizing: border-box;
    margin: 0;
    padding: 0;
}

body {
    font-family: Arial, Helvetica, sans-serif;
    background: #0b1020;
    color: #f4f5ff;
    min-height: 100vh;
}

.navbar {
    display: flex;
    justify-content: space-between;
    align-items: center;
    padding: 24px 8%;
    border-bottom: 1px solid #252c42;
}

.logo {
    font-size: 24px;
    font-weight: 800;
}

.logo span {
    color: #a78bfa;
}

.tagline {
    color: #a8afc7;
    font-size: 13px;
}

.container {
    max-width: 950px;
    margin: auto;
    padding: 70px 24px 30px;
}

.hero {
    text-align: center;
    margin-bottom: 42px;
}

.eyebrow {
    color: #a78bfa;
    letter-spacing: 3px;
    font-size: 12px;
    font-weight: bold;
    margin-bottom: 22px;
}

h1 {
    font-size: clamp(36px, 6vw, 60px);
    line-height: 1.12;
    letter-spacing: -2px;
}

h1 span {
    color: #a78bfa;
}

.description {
    max-width: 620px;
    margin: 22px auto 0;
    color: #a8afc7;
    line-height: 1.8;
}

.analyzer {
    background: #141b30;
    border: 1px solid #2a3450;
    border-radius: 18px;
    padding: 28px;
}

.analyzer label {
    display: block;
    font-weight: bold;
    margin-bottom: 14px;
}

textarea {
    width: 100%;
    min-height: 180px;
    resize: vertical;
    padding: 18px;
    border-radius: 12px;
    border: 1px solid #36415f;
    background: #0b1020;
    color: white;
    font: inherit;
    line-height: 1.7;
    outline: none;
}

textarea:focus {
    border-color: #a78bfa;
}

textarea::placeholder {
    color: #77809b;
}

.input-footer {
    display: flex;
    justify-content: space-between;
    align-items: center;
    gap: 16px;
    margin-top: 16px;
}

#charCount {
    color: #929bb6;
    font-size: 12px;
}

button {
    cursor: pointer;
    font: inherit;
}

#analyzeBtn {
    background: #a78bfa;
    color: #111329;
    border: none;
    border-radius: 9px;
    padding: 14px 20px;
    font-weight: bold;
}

#analyzeBtn:hover {
    background: #c4b5fd;
}

#analyzeBtn:disabled {
    opacity: 0.6;
    cursor: wait;
}

.examples {
    display: flex;
    flex-wrap: wrap;
    gap: 10px;
    align-items: center;
    margin-top: 22px;
}

.examples span {
    color: #929bb6;
    font-size: 12px;
}

.examples button {
    border: 1px solid #36415f;
    border-radius: 20px;
    padding: 8px 12px;
    background: transparent;
    color: #d7d9e8;
    font-size: 12px;
}

.examples button:hover {
    border-color: #a78bfa;
}

.result {
    margin-top: 28px;
    border-radius: 12px;
    padding: 24px;
    background: #0b1020;
    border: 1px solid #36415f;
}

.result-label {
    font-size: 11px;
    letter-spacing: 2px;
    color: #929bb6;
}

.result h2 {
    margin: 12px 0;
    font-size: 28px;
}

.result p:last-child {
    color: #b7bfd7;
    line-height: 1.7;
}

.result.positive {
    border-color: #34d399;
}

.result.positive h2 {
    color: #34d399;
}

.result.negative {
    border-color: #fb7185;
}

.result.negative h2 {
    color: #fb7185;
}

.result.error {
    border-color: #fb7185;
}

.result.error h2 {
    color: #fb7185;
}

.hidden {
    display: none;
}

.info {
    display: grid;
    grid-template-columns: repeat(3, 1fr);
    gap: 16px;
    margin-top: 24px;
}

.info-card {
    background: #141b30;
    border: 1px solid #2a3450;
    padding: 22px;
    border-radius: 14px;
    text-align: center;
    overflow-wrap: anywhere;
}

.info-card h3 {
    font-size: 19px;
    color: #a78bfa;
    margin-bottom: 10px;
}

.info-card p {
    font-size: 12px;
    color: #a8afc7;
    line-height: 1.6;
}

footer {
    text-align: center;
    margin-top: 45px;
    color: #77809b;
    font-size: 12px;
    line-height: 2;
}

@media (max-width: 600px) {
    .navbar {
        padding: 20px;
        align-items: flex-start;
        gap: 10px;
        flex-direction: column;
    }

    .container {
        padding: 48px 16px 24px;
    }

    .analyzer {
        padding: 18px;
    }

    .input-footer {
        align-items: stretch;
        flex-direction: column;
    }

    #analyzeBtn {
        width: 100%;
    }

    .info {
        grid-template-columns: 1fr;
    }
}

Writing static/style.css


In [6]:
%%writefile static/script.js

const reviewInput = document.getElementById("review");
const charCount = document.getElementById("charCount");
const analyzeBtn = document.getElementById("analyzeBtn");
const result = document.getElementById("result");

reviewInput.addEventListener("input", () => {
    charCount.textContent =
        `${reviewInput.value.length} / 10000 characters`;
});

function setReview(text) {
    reviewInput.value = text;
    reviewInput.dispatchEvent(new Event("input"));
    reviewInput.focus();
}

async function analyzeReview() {
    const review = reviewInput.value.trim();

    if (!review) {
        showError("Please enter a movie review first.");
        return;
    }

    analyzeBtn.disabled = true;
    analyzeBtn.textContent = "Analyzing...";
    result.className = "result hidden";

    try {
        const response = await fetch("/predict", {
            method: "POST",
            headers: {
                "Content-Type": "application/json"
            },
            body: JSON.stringify({ review })
        });

        const data = await response.json();

        if (!response.ok) {
            throw new Error(data.error || "Unable to analyze review.");
        }

        const sentiment = data.sentiment.toLowerCase();

        result.className = "result";

        if (sentiment === "positive" || sentiment === "negative") {
            result.classList.add(sentiment);
        }

        document.getElementById("sentiment").textContent =
            sentiment.toUpperCase();

        const scores = Object.entries(data.probabilities)
            .map(([label, value]) => `${label}: ${value}%`)
            .join(" | ");

        document.getElementById("probabilities").textContent =
            `Model probability estimates — ${scores}`;

    } catch (error) {
        showError(error.message);
    } finally {
        analyzeBtn.disabled = false;
        analyzeBtn.textContent = "Analyze sentiment →";
    }
}

function showError(message) {
    result.className = "result error";

    document.getElementById("sentiment").textContent = "Unable to analyze";

    document.getElementById("probabilities").textContent = message;
}

Writing static/script.js


In [9]:
%%writefile requirements.txt

Flask
joblib
scikit-learn
numpy
scipy
gunicorn

Overwriting requirements.txt


In [8]:
import subprocess
import sys
import time

server = subprocess.Popen(
    [sys.executable, "app.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(3)

if server.poll() is None:
    print("SentimentScope server started!")
    print("Open http://127.0.0.1:5000 in your browser.")
else:
    print(server.stdout.read())

SentimentScope server started!
Open http://127.0.0.1:5000 in your browser.


In [10]:
import os

print("Current folder:", os.getcwd())
print("\nFiles in this folder:")
print(os.listdir())

Current folder: C:\Users\saksh\jupyter

Files in this folder:
['.ipynb_checkpoints', 'app.py', 'first.ipynb', 'IMDB Dataset.csv', 'iris-dataset.ipynb', 'iris.ipynb', 'iris.zip', 'Mall_Customers.csv', 'Model_2.0.ipynb', 'Program_1_Basic_KMeans.ipynb', 'Program_2_Manual_KMeans.ipynb', 'requirements.txt', 'sa_BoW.ipynb', 'sa_TF-IDF.ipynb', 'sa_train-test-Copy1.ipynb', 'sa_train-test.ipynb', 'SentimentScope.ipynb', 'sentiment_model.pkl', 'static', 'templates', 'tfidf_vectorizer.pkl', 'titanic.csv', 'titanic.ipynb', 'Untitled.ipynb']
